In [1]:
import pandas as pd

## Using MSA version and converting. - Outdated, scroll down to using County version

In [2]:
hpi = pd.read_csv(
    "../../01_original_data/federal_housing_finance_agency/hpi_master.csv"
)
hpi = hpi[
    (hpi["level"] == "MSA") & (hpi["yr"] >= 1999) & (hpi["yr"] <= 2020)
]  # only keep data for msa level
hpi = hpi[["place_name", "place_id", "yr", "period", "index_nsa"]]
hpi = hpi.groupby(["place_id", "yr"], as_index=False).agg(
    {
        "place_name": "first",
        "place_id": "first",
        "yr": "first",
        "index_nsa": "mean",
    }
)

hpi["index_prev"] = hpi.groupby(["place_id"])["index_nsa"].shift(1)
hpi["hpi_change"] = hpi["index_nsa"] - hpi["index_prev"]

hpi["yr"] = pd.to_numeric(hpi["yr"], errors="coerce")
hpi["index_nsa"] = pd.to_numeric(hpi["index_nsa"], errors="coerce")

hpi["hpi_yoy"] = (hpi["index_nsa"] / hpi["index_prev"] - 1) * 100

hpi = hpi.dropna(subset=["hpi_yoy"])

hpi.sort_values(by=["place_id", "yr"])

,place_name,place_id,yr,index_nsa,index_prev,hpi_change,hpi_yoy
1,"Abilene, TX",10180,2000,115.7100,114.3075,1.4025,1.226954
2,"Abilene, TX",10180,2001,120.4200,115.7100,4.7100,4.070521
3,"Abilene, TX",10180,2002,123.8525,120.4200,3.4325,2.850440
4,"Abilene, TX",10180,2003,127.0975,123.8525,3.2450,2.620052
5,"Abilene, TX",10180,2004,132.7600,127.0975,5.6625,4.455241
...,...,...,...,...,...,...,...
9014,"Yuma, AZ",49740,2016,171.2425,164.9825,6.2600,3.794342
9015,"Yuma, AZ",49740,2017,177.8425,171.2425,6.6000,3.854183
9016,"Yuma, AZ",49740,2018,184.3200,177.8425,6.4775,3.642268
9017,"Yuma, AZ",49740,2019,193.8850,184.3200,9.5650,5.189345


In [3]:
xwalk = pd.read_csv(
    "../../01_original_data/federal_housing_finance_agency/list1_2023.csv"
)
xwalk.columns = (
    xwalk.columns.str.strip().str.lower().str.replace(r"\s+", "_", regex=True)
)
xwalk["county_fips5"] = (
    xwalk["fips_state_code"].astype(str).str.upper().str.strip().str.zfill(2)
    + xwalk["fips_county_code"].astype(str).str.zfill(3).str.strip()
)
xwalk = xwalk[
    [
        "cbsa_code",
        "cbsa_title",
        "county_fips5",
        "county/county_equivalent",
        "state_name",
    ]
]
xwalk["cbsa_code"] = xwalk["cbsa_code"].astype(str)
# xwalk.info()
xwalk

,cbsa_code,cbsa_title,county_fips5,county/county_equivalent,state_name
0,10100,"Aberdeen, SD",46013,Brown County,South Dakota
1,10100,"Aberdeen, SD",46045,Edmunds County,South Dakota
2,10140,"Aberdeen, WA",53027,Grays Harbor County,Washington
3,10180,"Abilene, TX",48059,Callahan County,Texas
4,10180,"Abilene, TX",48253,Jones County,Texas
...,...,...,...,...,...
1910,49700,"Yuba City, CA",06101,Sutter County,California
1911,49700,"Yuba City, CA",06115,Yuba County,California
1912,49740,"Yuma, AZ",04027,Yuma County,Arizona
1913,49780,"Zanesville, OH",39119,Muskingum County,Ohio


In [4]:
hpi_county_level = xwalk.merge(
    hpi,
    left_on=["cbsa_code"],  # , "nri_ver"],
    right_on=["place_id"],  # , "yr"],
    how="inner",
).dropna(subset="hpi_yoy")

hpi_county_level["yr"] = hpi_county_level["yr"].astype(int)
hpi_county_level

,cbsa_code,cbsa_title,county_fips5,county/county_equivalent,state_name,place_name,place_id,yr,index_nsa,index_prev,hpi_change,hpi_yoy
0,10180,"Abilene, TX",48059,Callahan County,Texas,"Abilene, TX",10180,2000,115.7100,114.3075,1.4025,1.226954
1,10180,"Abilene, TX",48059,Callahan County,Texas,"Abilene, TX",10180,2001,120.4200,115.7100,4.7100,4.070521
2,10180,"Abilene, TX",48059,Callahan County,Texas,"Abilene, TX",10180,2002,123.8525,120.4200,3.4325,2.850440
3,10180,"Abilene, TX",48059,Callahan County,Texas,"Abilene, TX",10180,2003,127.0975,123.8525,3.2450,2.620052
4,10180,"Abilene, TX",48059,Callahan County,Texas,"Abilene, TX",10180,2004,132.7600,127.0975,5.6625,4.455241
...,...,...,...,...,...,...,...,...,...,...,...,...
21960,49740,"Yuma, AZ",04027,Yuma County,Arizona,"Yuma, AZ",49740,2016,171.2425,164.9825,6.2600,3.794342
21961,49740,"Yuma, AZ",04027,Yuma County,Arizona,"Yuma, AZ",49740,2017,177.8425,171.2425,6.6000,3.854183
21962,49740,"Yuma, AZ",04027,Yuma County,Arizona,"Yuma, AZ",49740,2018,184.3200,177.8425,6.4775,3.642268
21963,49740,"Yuma, AZ",04027,Yuma County,Arizona,"Yuma, AZ",49740,2019,193.8850,184.3200,9.5650,5.189345


In [5]:
hpi[hpi["place_id"] == "10100"]

,place_name,place_id,yr,index_nsa,index_prev,hpi_change,hpi_yoy


In [6]:
# Not using this anymore because we have the new HPI in county level
# hpi_county_level.to_csv(
#     "../../02_processed_data/hpi_county_2000.csv",
#     index=False,
# )

In [7]:
hpi_county_level.columns

Index(['cbsa_code', 'cbsa_title', 'county_fips5', 'county/county_equivalent',
       'state_name', 'place_name', 'place_id', 'yr', 'index_nsa', 'index_prev',
       'hpi_change', 'hpi_yoy'],
      dtype='object')

## The new HPI at county level

In [8]:
NEW_HPI_NO_CONVERSION = pd.read_excel("../../01_original_data/hpi_at_county.xlsx")

In [9]:
NEW_HPI_NO_CONVERSION.iloc[4:8]

,HPI for Counties (All-Transactions Index)\nExperimental Indexes Showing Cumulative (Nominal) Annual Appreciation,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7
4,State,County,FIPS code,Year,Annual Change (%),HPI,HPI with 1990 base,HPI with 2000 base
5,AL,Autauga,01001,1986,NaN,100,95.54,70.81
6,AL,Autauga,01001,1987,-1.86,98.14,93.76,69.49
7,AL,Autauga,01001,1988,2.6,100.68,96.19,71.29


In [10]:
NEW_HPI_NO_CONVERSION.columns = [
    "state_name",
    "county/county_equivalent",
    "county_fips5",
    "yr",
    "hpi_change",
    "index_nsa",
    "drop1",
    "drop2",
]

In [11]:
NEW_HPI_NO_CONVERSION.iloc[4:8]

,state_name,county/county_equivalent,county_fips5,yr,hpi_change,index_nsa,drop1,drop2
4,State,County,FIPS code,Year,Annual Change (%),HPI,HPI with 1990 base,HPI with 2000 base
5,AL,Autauga,01001,1986,NaN,100,95.54,70.81
6,AL,Autauga,01001,1987,-1.86,98.14,93.76,69.49
7,AL,Autauga,01001,1988,2.6,100.68,96.19,71.29


In [12]:
NEW_HPI_NO_CONVERSION_ROI = NEW_HPI_NO_CONVERSION[
    [
        "state_name",
        "county/county_equivalent",
        "county_fips5",
        "yr",
        "hpi_change",
        "index_nsa",
    ]
].iloc[5:]

In [13]:
NEW_HPI_NO_CONVERSION_ROI.head()

,state_name,county/county_equivalent,county_fips5,yr,hpi_change,index_nsa
5,AL,Autauga,01001,1986,NaN,100
6,AL,Autauga,01001,1987,-1.86,98.14
7,AL,Autauga,01001,1988,2.6,100.68
8,AL,Autauga,01001,1989,4.3,105.02
9,AL,Autauga,01001,1990,-0.33,104.67


In [14]:
max(NEW_HPI_NO_CONVERSION_ROI["yr"])

2024

In [15]:
NEW_HPI_NO_CONVERSION_ROI.to_csv(
    "../../02_processed_data/hpi_county_2000_2024.csv",
    index=False,
)

### Just picking 2021 - 2024 HPI

In [ ]:
NEW_HPI_NO_CONVERSION_ROI_2021_2024 = NEW_HPI_NO_CONVERSION_ROI[
    NEW_HPI_NO_CONVERSION_ROI["yr"].between(2021, 2024)
]
NEW_HPI_NO_CONVERSION_ROI_2021_2024.to_csv(
    "../../02_processed_data/hpi_county_2021_2024.csv",
    index=False,
)
NEW_HPI_NO_CONVERSION_ROI_2021_2024

,state_name,county/county_equivalent,county_fips5,yr,hpi_change,index_nsa
40,AL,Autauga,01001,2021,8.04,207.76
41,AL,Autauga,01001,2022,14.03,236.89
42,AL,Autauga,01001,2023,9.35,259.05
43,AL,Autauga,01001,2024,8.2,280.3
88,AL,Baldwin,01003,2021,13.41,474.46
...,...,...,...,...,...,...
103269,WY,Washakie,56043,2024,1.88,401.4
103295,WY,Weston,56045,2021,4.97,246.69
103296,WY,Weston,56045,2022,6.15,261.87
103297,WY,Weston,56045,2023,1.32,265.32
